# Project 22: Restaurant Orders & Menu Performance Analysis

**Dataset:** `22_restaurant_orders_1.csv`  
**Libraries used:** `pandas` and `numpy` only

**Name:** ____________________   **Roll No:** ____________

### How this notebook is written
- All column names are used **exactly as they are in the CSV file** (nothing is renamed and no new columns are added to the table).
- Where the assignment asks for a new value (weekday, Item_Revenue, time category) it is stored in a separate variable, not in the table.
- The assignment names and the CSV names match like this: Item_Quantity = `Quantity`, Item_Price = `Unit_Price`, takeaway = `Pickup`. The CSV has no Order_DateTime column, so we use `Order_Date` (date) and `Order_Hour` (hour).

## Task 1: Load and inspect the dataset

In [1]:
import pandas as pd
import numpy as np

pd.options.display.float_format = '{:.2f}'.format   # only shows 2 decimal places

df = pd.read_csv("22_restaurant_orders_1.csv")
df.head()

,Order_ID,Order_Date,Restaurant_Name,Cuisine,Dish_Name,Category,Quantity,Unit_Price,Order_Hour,Delivery_Mode
0,FO000000,2025-08-31,Pizza Corner,Healthy,Pasta,Main Course,1.00,336.58,17,Dine-in
1,FO000001,2025-01-16,Green Bowl,Italian,Salad,Beverage,5.00,789.81,12,Dine-in
2,FO000002,2026-05-31,Tandoori House,Indian,Burger,Dessert,5.00,717.33,19,Delivery
3,FO000003,2025-01-05,Green Bowl,Healthy,Paneer Curry,Snack,2.00,484.37,15,Delivery
4,FO000004,2025-03-26,Pizza Corner,Indian,Burger,Dessert,5.00,482.80,8,Dine-in


In [2]:
print("Rows and columns:", df.shape)
print()
print("Columns:", list(df.columns))

Rows and columns: (862, 10)

Columns: ['Order_ID', 'Order_Date', 'Restaurant_Name', 'Cuisine', 'Dish_Name', 'Category', 'Quantity', 'Unit_Price', 'Order_Hour', 'Delivery_Mode']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 862 entries, 0 to 861
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         862 non-null    str    
 1   Order_Date       862 non-null    str    
 2   Restaurant_Name  862 non-null    str    
 3   Cuisine          862 non-null    str    
 4   Dish_Name        862 non-null    str    
 5   Category         862 non-null    str    
 6   Quantity         841 non-null    float64
 7   Unit_Price       840 non-null    float64
 8   Order_Hour       862 non-null    int64  
 9   Delivery_Mode    862 non-null    str    
dtypes: float64(2), int64(1), str(7)
memory usage: 67.5 KB


In [4]:
df.describe()

,Quantity,Unit_Price,Order_Hour
count,841.00,840.00,862.00
mean,2.97,490.30,15.29
std,1.43,233.81,4.65
min,1.00,81.02,8.00
25%,2.00,290.42,11.00
50%,3.00,499.21,15.00
75%,4.00,687.83,19.00
max,5.00,897.47,23.00


In [5]:
for col in ["Restaurant_Name", "Cuisine", "Dish_Name", "Category", "Delivery_Mode"]:
    print(col, "->", df[col].unique().tolist())

Restaurant_Name -> ['Pizza Corner', 'Green Bowl', 'Tandoori House', 'Spice Hub', 'Urban Cafe']
Cuisine -> ['Healthy', 'Italian', 'Indian', 'Mexican', 'Chinese']
Dish_Name -> ['Pasta', 'Salad', 'Burger', 'Paneer Curry', 'Biryani', 'Noodles', 'Tacos', 'Pizza']
Category -> ['Main Course', 'Beverage', 'Dessert', 'Snack']
Delivery_Mode -> ['Dine-in', 'Delivery', 'Pickup']


**Observations**
- 862 rows and 10 columns. Each row is one order of one dish.
- 5 restaurants, 8 dishes, 4 categories, 3 order types (Dine-in, Pickup, Delivery).
- `Quantity` and `Unit_Price` have missing values (fewer than 862 non-null in info()).
- There is no customer ID column in this file.

## Task 2: Missing values and duplicate Order_ID

In [6]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
Order_ID            0
Order_Date          0
Restaurant_Name     0
Cuisine             0
Dish_Name           0
Category            0
Quantity           21
Unit_Price         22
Order_Hour          0
Delivery_Mode       0
dtype: int64


In [7]:
print("Duplicate Order_ID          :", df["Order_ID"].duplicated().sum())
print("Fully duplicate rows        :", df.duplicated().sum())

Duplicate Order_ID          : 12
Fully duplicate rows        : 12


In [8]:
# the duplicate records
df[df["Order_ID"].duplicated(keep=False)].sort_values("Order_ID")

,Order_ID,Order_Date,Restaurant_Name,Cuisine,Dish_Name,Category,Quantity,Unit_Price,Order_Hour,Delivery_Mode
39,FO000039,2025-10-27,Green Bowl,Healthy,Pizza,Main Course,4.00,320.61,11,Dine-in
854,FO000039,2025-10-27,Green Bowl,Healthy,Pizza,Main Course,4.00,320.61,11,Dine-in
66,FO000066,2025-07-28,Spice Hub,Chinese,Tacos,Dessert,2.00,491.92,20,Pickup
855,FO000066,2025-07-28,Spice Hub,Chinese,Tacos,Dessert,2.00,491.92,20,Pickup
110,FO000110,2025-04-25,Pizza Corner,Indian,Paneer Curry,Dessert,2.00,545.11,8,Delivery
852,FO000110,2025-04-25,Pizza Corner,Indian,Paneer Curry,Dessert,2.00,545.11,8,Delivery
260,FO000260,2026-02-05,Spice Hub,Indian,Biryani,Snack,3.00,216.28,10,Dine-in
857,FO000260,2026-02-05,Spice Hub,Indian,Biryani,Snack,3.00,216.28,10,Dine-in
357,FO000357,2025-01-13,Green Bowl,Chinese,Salad,Dessert,2.00,581.68,15,Dine-in
851,FO000357,2025-01-13,Green Bowl,Chinese,Salad,Dessert,2.00,581.68,15,Dine-in


**Observations**
- `Quantity` has 21 missing values and `Unit_Price` has 22 missing values.
- 12 Order_IDs are repeated and in every case the **whole row is identical**, so these are repeated entries of the same order, not new orders.

## Task 3: Clean missing Quantity and Unit_Price
**Duplicates:** the 12 repeated rows are exact copies, so we drop them. Otherwise the same order would be counted twice in orders, items and revenue. (This is the only line that removes rows – delete it if your teacher wants all 862 rows kept.)

In [9]:
df = df.drop_duplicates()
print("Rows now:", len(df))
print("Order_ID unique now?", df["Order_ID"].is_unique)

Rows now: 850
Order_ID unique now? True


**Method for the missing values**
- **Unit_Price:** filled with the **median price of the same dish**. Median is used because prices are spread widely (about 80 to 900) and the median is not pulled by very high or very low prices. The dish is used because each dish has a slightly different median price.
- **Quantity:** filled with the **median of the whole column** (3). Quantity is always 1 to 5, so the median is a sensible typical value.
- Rows are not deleted because only about 2.5% of values are missing and the other columns of those rows are useful.

In [10]:
print("Median price of each dish:")
print(df.groupby("Dish_Name")["Unit_Price"].median())
print()
print("Median Quantity:", df["Quantity"].median())

Median price of each dish:
Dish_Name
Biryani        429.68
Burger         487.33
Noodles        474.54
Paneer Curry   541.25
Pasta          499.09
Pizza          577.39
Salad          448.40
Tacos          494.67
Name: Unit_Price, dtype: float64

Median Quantity: 3.0


In [11]:
print("Missing before:")
print(df[["Quantity", "Unit_Price"]].isnull().sum())

# fill Unit_Price with the median of the same dish
for dish in df["Dish_Name"].unique():
    dish_median = df.loc[df["Dish_Name"] == dish, "Unit_Price"].median()
    df.loc[(df["Dish_Name"] == dish) & (df["Unit_Price"].isnull()), "Unit_Price"] = dish_median

# fill Quantity with the median of the column
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

print()
print("Missing after:")
print(df[["Quantity", "Unit_Price"]].isnull().sum())

Missing before:
Quantity      21
Unit_Price    21
dtype: int64

Missing after:
Quantity      0
Unit_Price    0
dtype: int64


## Task 4: Convert to datetime, extract hour and weekday
`Order_Date` is converted to datetime. The hour is already given in `Order_Hour`; we also build the full date-time in a variable `order_datetime` and take hour and weekday from it using `.dt.hour` and `.dt.day_name()`.

In [12]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
print(df["Order_Date"].dtype)

order_datetime = df["Order_Date"] + pd.to_timedelta(df["Order_Hour"], unit="h")

hour = order_datetime.dt.hour
weekday = order_datetime.dt.day_name()

print(pd.DataFrame({"Order_ID": df["Order_ID"], "order_datetime": order_datetime, "hour": hour, "weekday": weekday}).head())

datetime64[us]
   Order_ID      order_datetime  hour    weekday
0  FO000000 2025-08-31 17:00:00    17     Sunday
1  FO000001 2025-01-16 12:00:00    12   Thursday
2  FO000002 2026-05-31 19:00:00    19     Sunday
3  FO000003 2025-01-05 15:00:00    15     Sunday
4  FO000004 2025-03-26 08:00:00     8  Wednesday


## Task 5: Total orders, total items sold, total revenue

In [13]:
total_orders = df["Order_ID"].nunique()
total_items = df["Quantity"].sum()
total_revenue = (df["Quantity"] * df["Unit_Price"]).sum()

print("Total orders     :", total_orders)
print("Total items sold :", total_items)
print("Total revenue    :", round(total_revenue, 2))

Total orders     : 850
Total items sold : 2525.0
Total revenue    : 1241346.8


## Task 6: Row-wise Item_Revenue
Item_Revenue = Quantity × Unit_Price. It is kept in the variable `item_revenue` (same row order as the table).

In [14]:
item_revenue = df["Quantity"] * df["Unit_Price"]

print(pd.DataFrame({"Order_ID": df["Order_ID"], "Quantity": df["Quantity"], "Unit_Price": df["Unit_Price"], "Item_Revenue": item_revenue}).head())

   Order_ID  Quantity  Unit_Price  Item_Revenue
0  FO000000      1.00      336.58        336.58
1  FO000001      5.00      789.81       3949.05
2  FO000002      5.00      717.33       3586.65
3  FO000003      2.00      484.37        968.74
4  FO000004      5.00      482.80       2414.00


Every Order_ID is unique and has one dish, so the **bill of an order = its Item_Revenue**.

## Task 7: Most frequently ordered dishes
The assignment says top 10, but the menu has only **8 dishes**, so all 8 are shown.

In [15]:
top_dishes_count = df["Dish_Name"].value_counts().head(10)
print(top_dishes_count)

Dish_Name
Pasta           116
Pizza           112
Biryani         109
Tacos           109
Noodles         106
Salad           101
Burger          101
Paneer Curry     96
Name: count, dtype: int64


In [16]:
# orders, units sold and revenue for each dish
dish_table = pd.DataFrame({
    "Orders": df["Dish_Name"].value_counts(),
    "Units_Sold": df.groupby("Dish_Name")["Quantity"].sum(),
    "Revenue": item_revenue.groupby(df["Dish_Name"]).sum()
})
dish_table["Revenue_per_Order"] = dish_table["Revenue"] / dish_table["Orders"]
dish_table["Revenue_%"] = dish_table["Revenue"] / dish_table["Revenue"].sum() * 100
dish_table["Revenue_Rank"] = dish_table["Revenue"].rank(ascending=False)
dish_table.sort_values("Orders", ascending=False)

,Orders,Units_Sold,Revenue,Revenue_per_Order,Revenue_%,Revenue_Rank
Dish_Name,,,,,,
Pasta,116,358.00,183360.76,1580.70,14.77,1.00
Pizza,112,326.00,177193.58,1582.09,14.27,2.00
Biryani,109,323.00,140478.98,1288.80,11.32,8.00
Tacos,109,311.00,154729.59,1419.54,12.46,3.00
Noodles,106,334.00,153561.08,1448.69,12.37,4.00
Burger,101,302.00,143898.76,1424.74,11.59,6.00
Salad,101,299.00,144457.06,1430.27,11.64,5.00
Paneer Curry,96,272.00,143666.98,1496.53,11.57,7.00


**Observations**
- **Pasta** (116 orders) and **Pizza** (112) are the most ordered and also the top 2 in revenue.
- **Paneer Curry** has the fewest orders (96).
- **Biryani** is 3rd in orders but last (8th) in revenue because it has the lowest price.

## Task 8: Rank menu categories by total revenue

In [17]:
category_revenue = item_revenue.groupby(df["Category"]).sum().sort_values(ascending=False)

category_table = pd.DataFrame({
    "Revenue": category_revenue,
    "Share_%": category_revenue / category_revenue.sum() * 100
})
category_table["Rank"] = range(1, len(category_table) + 1)
category_table

,Revenue,Share_%,Rank
Category,,,
Snack,353475.25,28.48,1
Dessert,305760.55,24.63,2
Beverage,299819.78,24.15,3
Main Course,282291.21,22.74,4


**Observation:** Snack earns the most (about 28.5%) and Main Course the least (about 22.7%). The four categories are fairly close to each other.

## Task 9: Average bill – dine-in, takeaway (Pickup), delivery

In [18]:
avg_bill = item_revenue.groupby(df["Delivery_Mode"]).mean().sort_values(ascending=False)
print("Average bill by order type:")
print(avg_bill)
print()
print("Number of orders by order type:")
print(df["Delivery_Mode"].value_counts())

Average bill by order type:
Delivery_Mode
Dine-in    1523.72
Pickup     1460.60
Delivery   1391.97
dtype: float64

Number of orders by order type:
Delivery_Mode
Dine-in     304
Delivery    282
Pickup      264
Name: count, dtype: int64


**Observation:** Dine-in has the highest average bill (about 1524), then Pickup (about 1461), and Delivery the lowest (about 1392).

## Task 10: Peak ordering hours and busiest day

In [19]:
orders_by_hour = hour.value_counts()
print("Top 5 busiest hours:")
print(orders_by_hour.head(5))
print()
print("Peak hour:", orders_by_hour.idxmax(), "with", orders_by_hour.max(), "orders")

Top 5 busiest hours:
15    66
9     62
8     59
12    58
19    58
Name: count, dtype: int64

Peak hour: 15 with 66 orders


In [20]:
orders_by_day = weekday.value_counts()
print("Orders by weekday:")
print(orders_by_day)
print()
print("Busiest day (orders) :", orders_by_day.idxmax())

revenue_by_day = item_revenue.groupby(weekday).sum()
print("Busiest day (revenue):", revenue_by_day.idxmax())
print()
print(revenue_by_day.sort_values(ascending=False))

Orders by weekday:
Wednesday    138
Thursday     132
Monday       129
Saturday     119
Friday       118
Sunday       107
Tuesday      107
Name: count, dtype: int64

Busiest day (orders) : Wednesday
Busiest day (revenue): Saturday

Saturday    192428.62
Thursday    191829.44
Monday      191702.54
Wednesday   181903.94
Friday      168396.48
Sunday      160763.16
Tuesday     154322.61
dtype: float64


**Observations**
- Peak hours: **15 (3 PM)**, **9** and **8** (morning).
- **Wednesday** has the most orders; **Saturday** gives the most revenue.

## Task 11: List for top dishes, tuple for fixed meal categories
A list can be changed, so it is used for top dishes. A tuple cannot be changed, so it is used for the fixed meal categories.

In [21]:
top_dishes = list(top_dishes_count.head(5).index)
print("Top 5 dishes (list):", top_dishes)

top_dishes.append("Test Dish")      # a list can be changed
print("After append:", top_dishes)
top_dishes.remove("Test Dish")
print("After remove:", top_dishes)

meal_categories = ("Beverage", "Dessert", "Main Course", "Snack")
print()
print("Meal categories (tuple):", meal_categories)

Top 5 dishes (list): ['Pasta', 'Pizza', 'Biryani', 'Tacos', 'Noodles']
After append: ['Pasta', 'Pizza', 'Biryani', 'Tacos', 'Noodles', 'Test Dish']
After remove: ['Pasta', 'Pizza', 'Biryani', 'Tacos', 'Noodles']

Meal categories (tuple): ('Beverage', 'Dessert', 'Main Course', 'Snack')


In [22]:
# a tuple cannot be changed
try:
    meal_categories[0] = "Starter"
except TypeError as e:
    print("Error:", e)

Error: 'tuple' object does not support item assignment


## Task 12: Set of unique IDs and a function for the total bill
The CSV has **no customer ID column**, so unique customers cannot be found. The set is used on the IDs that exist: `Order_ID`, `Restaurant_Name` and `Dish_Name`.

In [23]:
unique_order_ids = set(df["Order_ID"])
unique_restaurants = set(df["Restaurant_Name"])
unique_dishes = set(df["Dish_Name"])

print("Unique Order_IDs:", len(unique_order_ids))
print("Restaurants     :", unique_restaurants)
print("Dishes          :", unique_dishes)

Unique Order_IDs: 850
Restaurants     : {'Spice Hub', 'Tandoori House', 'Urban Cafe', 'Green Bowl', 'Pizza Corner'}
Dishes          : {'Pasta', 'Biryani', 'Pizza', 'Paneer Curry', 'Tacos', 'Salad', 'Noodles', 'Burger'}


In [24]:
def calculate_total_bill(order_id):
    order = df[df["Order_ID"] == order_id]
    if len(order) == 0:
        return "Order not found"
    return round((order["Quantity"] * order["Unit_Price"]).sum(), 2)

print("Bill of FO000000:", calculate_total_bill("FO000000"))
print("Bill of FO000001:", calculate_total_bill("FO000001"))
print("Bill of FO999999:", calculate_total_bill("FO999999"))

Bill of FO000000: 336.58
Bill of FO000001: 3949.05
Bill of FO999999: Order not found


## Task 13: NumPy – mean, median, maximum bill

In [25]:
bills = item_revenue.to_numpy()

print("Mean bill   :", round(np.mean(bills), 2))
print("Median bill :", round(np.median(bills), 2))
print("Maximum bill:", round(np.max(bills), 2))

Mean bill   : 1460.41
Median bill : 1181.16
Maximum bill: 4487.35


The mean (about 1460) is higher than the median (about 1181), so a few big bills pull the average up.

## Task 14: Unusually large bills (IQR method) and order-time categories
IQR = Q3 − Q1, upper limit = Q3 + 1.5 × IQR, lower limit = Q1 − 1.5 × IQR. A bill outside the limits is unusual.

In [26]:
q1 = np.percentile(bills, 25)
q3 = np.percentile(bills, 75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

print("Q1 =", round(q1, 2))
print("Q3 =", round(q3, 2))
print("IQR =", round(iqr, 2))
print("Lower limit =", round(lower_limit, 2))
print("Upper limit =", round(upper_limit, 2))

Q1 = 625.98
Q3 = 2149.83
IQR = 1523.86
Lower limit = -1659.81
Upper limit = 4435.61


In [27]:
is_outlier = (item_revenue > upper_limit) | (item_revenue < lower_limit)
print("Unusually large bills:", is_outlier.sum())
print()
print("Their bill values:")
print(item_revenue[is_outlier])
print()
df[is_outlier]

Unusually large bills: 3

Their bill values:
48    4484.55
425   4441.65
454   4487.35
dtype: float64



,Order_ID,Order_Date,Restaurant_Name,Cuisine,Dish_Name,Category,Quantity,Unit_Price,Order_Hour,Delivery_Mode
48,FO000048,2025-04-09,Pizza Corner,Mexican,Pizza,Snack,5.00,896.91,17,Delivery
425,FO000425,2025-01-31,Spice Hub,Indian,Paneer Curry,Snack,5.00,888.33,12,Delivery
454,FO000454,2025-11-10,Urban Cafe,Mexican,Salad,Beverage,5.00,897.47,20,Pickup


Only **3 orders** are unusually large. All three have Quantity 5 and a price of about 890, so the big bill comes from both values being high.

**Order-time categories:** Morning = 5 to 11, Afternoon = 12 to 16, Evening = 17 to 20, Night = 21 to 4.

In [28]:
def get_time_category(h):
    if h >= 5 and h <= 11:
        return "Morning"
    elif h >= 12 and h <= 16:
        return "Afternoon"
    elif h >= 17 and h <= 20:
        return "Evening"
    else:
        return "Night"

time_category = df["Order_Hour"].apply(get_time_category)

time_table = pd.DataFrame({
    "Orders": time_category.value_counts(),
    "Revenue": item_revenue.groupby(time_category).sum(),
    "Average_Bill": item_revenue.groupby(time_category).mean()
})
time_table["Revenue_%"] = time_table["Revenue"] / time_table["Revenue"].sum() * 100
time_table.loc[["Morning", "Afternoon", "Evening", "Night"]]

,Orders,Revenue,Average_Bill,Revenue_%
Order_Hour,,,,
Morning,227,336623.53,1482.92,27.12
Afternoon,268,408891.32,1525.71,32.94
Evening,209,294524.36,1409.21,23.73
Night,146,201307.58,1378.82,16.22


**Observation:** Afternoon gives the most revenue (about 33%) and the highest average bill. Night gives the least (about 16%) but it also covers only 3 hours (21, 22, 23).

## Task 15: MenuItem class and five recommendations
Each dish becomes a `MenuItem` with its name, its most common category and its median price.

In [29]:
class MenuItem:
    def __init__(self, name, category, price):
        self.name = name
        self.category = category
        self.price = price

    def show(self):
        print(self.name, "|", self.category, "| price:", round(self.price, 2))


menu = []
for dish in df["Dish_Name"].unique():
    dish_rows = df[df["Dish_Name"] == dish]
    category = dish_rows["Category"].value_counts().index[0]
    price = dish_rows["Unit_Price"].median()
    menu.append(MenuItem(dish, category, price))

for item in menu:
    item.show()

Pasta | Snack | price: 499.09
Salad | Beverage | price: 448.4
Burger | Dessert | price: 487.33
Paneer Curry | Beverage | price: 541.24
Biryani | Snack | price: 429.68
Noodles | Dessert | price: 474.54
Tacos | Dessert | price: 494.67
Pizza | Main Course | price: 577.39


In [30]:
print("Average revenue per order of the whole menu:", round(dish_table["Revenue_per_Order"].mean(), 2))
print()
dish_table.sort_values("Orders", ascending=False)

Average revenue per order of the whole menu: 1458.92



,Orders,Units_Sold,Revenue,Revenue_per_Order,Revenue_%,Revenue_Rank
Dish_Name,,,,,,
Pasta,116,358.00,183360.76,1580.70,14.77,1.00
Pizza,112,326.00,177193.58,1582.09,14.27,2.00
Biryani,109,323.00,140478.98,1288.80,11.32,8.00
Tacos,109,311.00,154729.59,1419.54,12.46,3.00
Noodles,106,334.00,153561.08,1448.69,12.37,4.00
Burger,101,302.00,143898.76,1424.74,11.59,6.00
Salad,101,299.00,144457.06,1430.27,11.64,5.00
Paneer Curry,96,272.00,143666.98,1496.53,11.57,7.00


### Five menu-planning recommendations

**1. Increase the price of Biryani or sell it as a combo.**  
It is the 3rd most ordered dish (109 orders) but last in revenue (rank 8). It has the lowest price (about 430) and the lowest revenue per order (about 1289 against a menu average of about 1459). Customers like it, so a small price increase or a combo with a snack or drink can raise revenue.

**2. Keep Pasta and Pizza as the main dishes.**  
They are number 1 and 2 in both orders and revenue and together give about 29% of the revenue. Pizza has the highest price but is still 2nd in orders, so price is not stopping customers. Show them first on the menu.

**3. Promote Paneer Curry instead of removing it.**  
It has the fewest orders (96) and units (272), but its revenue per order (about 1497) is above the menu average. The dish is not weak; it needs more orders, so a special offer can help.

**4. Plan staff and offers by time of day.**  
Peak hours are 3 PM, 9 AM and 8 AM, and Afternoon gives the most revenue (about 33%). Keep more staff at these times. Night (about 16%) and the quieter days (Tuesday and Sunday) can use offers like late-night combos.

**5. Increase the bill of Delivery and Pickup orders.**  
Dine-in bills are highest (about 1524) and Delivery lowest (about 1392). "Add a snack" suggestions or a minimum-order combo can reduce the gap. Snack is also the top category (28.5% of revenue), so it is a good item to suggest.

**Limitation:** the differences between categories and order types are small compared with how much bills vary from order to order, and the data looks like practice data (the same dish appears under different categories and cuisines). So these are suggestions to test, not guaranteed results.

## Save the cleaned data and a short summary

In [31]:
df.to_csv("22_restaurant_orders_cleaned.csv", index=False)

file = open("22_analysis_summary.txt", "w")
file.write("RESTAURANT ORDERS - SUMMARY\n")
file.write("---------------------------\n")
file.write("Total orders        : " + str(total_orders) + "\n")
file.write("Total items sold    : " + str(total_items) + "\n")
file.write("Total revenue       : " + str(round(total_revenue, 2)) + "\n")
file.write("Mean bill           : " + str(round(np.mean(bills), 2)) + "\n")
file.write("Median bill         : " + str(round(np.median(bills), 2)) + "\n")
file.write("Maximum bill        : " + str(round(np.max(bills), 2)) + "\n")
file.write("Most ordered dish   : " + top_dishes_count.index[0] + "\n")
file.write("Top category        : " + category_revenue.index[0] + "\n")
file.write("Highest avg bill    : " + avg_bill.index[0] + "\n")
file.write("Peak hour           : " + str(orders_by_hour.idxmax()) + "\n")
file.write("Busiest weekday     : " + orders_by_day.idxmax() + "\n")
file.write("Unusual large bills : " + str(is_outlier.sum()) + "\n")
file.close()

print(open("22_analysis_summary.txt").read())

RESTAURANT ORDERS - SUMMARY
---------------------------
Total orders        : 850
Total items sold    : 2525.0
Total revenue       : 1241346.8
Mean bill           : 1460.41
Median bill         : 1181.16
Maximum bill        : 4487.35
Most ordered dish   : Pasta
Top category        : Snack
Highest avg bill    : Dine-in
Peak hour           : 15
Busiest weekday     : Wednesday
Unusual large bills : 3



## Conclusion
- After removing 12 repeated rows there are **850 orders**, **2525 items sold** and a total revenue of about **1,241,347**.
- **Pasta** and **Pizza** are the best dishes, **Snack** is the top category and **Dine-in** has the highest average bill.
- Busiest times are **3 PM and 8–9 AM**; the busiest day is **Wednesday** by orders and **Saturday** by revenue.
- Only **3** bills are unusually large.